# Восстановление и очистка аудио

Этот блокнот запускает русский Gradio-интерфейс. Google Drive и ключи API не нужны.

Перед запуском выбери в меню Colab: **Среда выполнения → Сменить среду выполнения → T4 GPU**. Затем выполняй ячейки сверху вниз. Третья кодовая ячейка заранее скачает FlashSR.

In [ ]:
# Проверка видеокарты и системных программ
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=True)
else:
    print("Видеокарта не найдена. Включи T4 GPU в настройках среды выполнения.")

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "ffmpeg", "git-lfs", "espeak-ng"],
    check=True,
)
subprocess.run(["git", "lfs", "install"], check=True)

if not shutil.which("uv") and not shutil.which("/root/.local/bin/uv"):
    subprocess.run(
        ["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"],
        check=True,
    )

print("Системная подготовка завершена.")

In [ ]:
# Загрузка проекта и установка только лёгкого интерфейса
import os
import shutil
import subprocess
from pathlib import Path

repo = Path("/content/audio-restoration-colab")
repo_url = "https://github.com/egor125552/audio-restoration-colab.git"

if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)

uv = shutil.which("uv") or "/root/.local/bin/uv"
print("1 из 3: подготавливаю Python 3.11…", flush=True)
subprocess.run([uv, "python", "install", "3.11"], check=True)
print("2 из 3: создаю отдельную среду…", flush=True)
subprocess.run(
    [uv, "venv", "--allow-existing", "--python", "3.11", str(repo / ".venv")],
    check=True,
)
print("3 из 3: устанавливаю интерфейс…", flush=True)
subprocess.run(
    [uv, "pip", "install", "--python", str(repo / ".venv/bin/python"), str(repo)],
    check=True,
)

os.chdir(repo)
print("Интерфейс установлен.")

In [ ]:
# Предварительная загрузка FlashSR. Обычно занимает несколько минут.
import subprocess

print("Начинаю загрузку FlashSR: около 3,2 ГБ весов.", flush=True)
subprocess.run(
    [
        "/content/audio-restoration-colab/scripts/prepare_backend.sh",
        "flashsr",
        "/content/audio-restoration-models",
    ],
    check=True,
)
print("FlashSR полностью скачана и установлена.", flush=True)

In [ ]:
# Запуск. Открой появившуюся временную ссылку Gradio.
import os
import re
import subprocess
import time
from pathlib import Path

log_path = Path("/content/audio-restoration-gradio.log")
old_process = globals().get("gradio_process")
if old_process is not None and old_process.poll() is None:
    old_process.terminate()
    old_process.wait(timeout=15)
log_file = log_path.open("w", encoding="utf-8")
gradio_process = subprocess.Popen(
    [
        "/content/audio-restoration-colab/.venv/bin/python",
        "-m",
        "audio_restoration_colab.app",
        "--share",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)

print("Запускаю интерфейс…", flush=True)
for _ in range(180):
    time.sleep(1)
    log_text = log_path.read_text(encoding="utf-8", errors="replace")
    match = re.search(r"https://[^\s]+\.gradio\.live", log_text)
    if match:
        print("Интерфейс готов:", match.group(0))
        break
    if gradio_process.poll() is not None:
        raise RuntimeError("Gradio завершился с ошибкой:\n" + log_text[-3000:])
else:
    gradio_process.terminate()
    raise RuntimeError("Gradio не выдал ссылку за 3 минуты:\n" + log_text[-3000:])